# Embedded-policy runs: case comparison

Reads the per-rank `drlrec*.bin` files written by `drl/pol_IO.f` for a **list**
of runs, overlays each category of result in one panel, and saves two things to
`data/results/<solver_case>/<run_name>/drl/` as `.mat`: the full per-agent
record (section 8.1, GB-scale) and the plot-ready arrays behind the four
figures (section 8.2, kB-scale, in `processed_data/`).

`drlrec_analysis.ipynb` is the single-case deep dive; this notebook is the
comparison, laid out the way `deterministic.ipynb` is: a case table at the top,
then one figure per category with every case drawn into it.

**Conventions**

* `obs` is the *raw* wall-tangential/normal fluctuation at the sensing plane in
  solver units — the exported network divides by `u_tau` internally, so what is
  recorded is what the flow presented. It is divided by `u_tau` here for the plots.
* `act` is the actuation velocity as applied, **before** the zero-net-mass-flux
  correction. `pol_eval` has already applied the action scale, so nothing is
  rescaled here.
* The reward slots mean different things per mode: `(tau_w, |p'v|, 0.5|v^3|)` in
  `net_gain`, `(dUdy, 0, 0)` in `dudy`. The mode is read from `drl_policy.in`
  and the decomposition follows it.
* `R_tau`, `R_pw`, `R_v3` are the **unweighted** terms; only `R_tot` carries
  `alpha`, `beta`, `gamma`.
* Wall points that no policy claims (`ipol == 0`) never act and are dropped by
  default — on the wing that is half of them. Pass `agents='all'` to keep them.

## Environment Setup

In [ ]:
import os
import copy
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

from postlib.plot import plt_setUp, make_style, colorplate as cc
from postlib import drlrec as dr

plt_setUp()
run_path = "../runs/"
fig_path = "Figs/"
os.makedirs(fig_path, exist_ok=True)

# --- Style ---
# Categorical slots in fixed order, never cycled, each with its own marker so
# identity survives colour-vision deficiency and greyscale printing.
STYLE_A = make_style(cc.blue,      0.9, 'o', 'A')
STYLE_B = make_style(cc.red,       0.9, 'X', 'B')
STYLE_C = make_style(cc.deepgreen, 0.9, 'D', 'C')
STYLE_D = make_style(cc.deeppurple, 0.9, 's', 'D')

SEQ = 'Blues'      # magnitude: one hue, light -> dark
DIV = 'RdBu_r'     # signed: two hues about a neutral midpoint

# Axis labels of the two observation components, used by every panel below.
obs_lab = [r"$u_t^+$", r"$v_n^+$"]

## Add your case

One row per run: `[runs/ folder, label, style]`. `overrides` is only needed for
runs staged without a `current_conf.yml`; everything resolvable is resolved from
`drl_policy.in`, the `.par`/`.rea`, and the record headers.

In [ ]:
SAVE_IMG = True
HEAD = 'DRLREC'

case_tuple = [
    ['solo_timing_cmp', 'phill DDPG', STYLE_A],
    ['small_wing_nes',  'wing NES',   STYLE_B],
]

# {case: {'utau': ..., 'nu': ..., 'dudy_ref': ...}} -- beats every other source
overrides = {}

# Averaging window, in the same units as the time axis (t+ where resolvable).
TRANS_TIME, EVAL_TIME = 500.0, 1500.0

case_list = [c[0] for c in case_tuple]
case_name = [c[1] for c in case_tuple]
case_style = [c[2] for c in case_tuple]

# Binning of the plot-ready products. The figures below and the .mat files
# section 8 writes are built from the *same* objects, so these are set once.
N_MESH = 80          # observation joint-PDF bins per axis
N_BIN = 50           # control-law bins per axis
N_AGENT_SHOW = 3     # action traces drawn per case
MIN_COUNT = 1        # control-law bins holding fewer samples are NaN

In [ ]:
case_dict = dr.read_drlrec_cases(run_path, case_list, overrides=overrides,
                                 agents='active', verbose=True)

# A t+ axis and a raw-time axis must not share one plot; say so rather than
# drawing a figure whose x axis means two different things.
units = {e['envs'][0]['series']['tp_unit'] for e in case_dict.values()}
TP_UNIT = units.pop() if len(units) == 1 else None
if TP_UNIT is None:
    print("[WARN] cases disagree on the time unit; give the missing u_tau/nu "
          "in `overrides` before reading the overlays.")
XLABEL = r"$t^+$" if TP_UNIT == 't+' else r"$t$"
XMAX = max(e['envs'][0]['series']['tp'][-1] for e in case_dict.values())

### Where the numbers came from

Every scalar the reward decomposition rests on, and which file won for it.

In [ ]:
prov = pd.DataFrame([{
    'label': case_name[il],
    'case': case,
    'solver_case': e['solver_case'],
    'mode': e['scale']['reward_mode'],
    'nu': e['scale']['nu'],
    'u_tau': e['scale']['utau'],
    'dUdy_ref': e['scale']['dudy_ref'],
    'ref': e['scale']['ref'],
    'a/b/g': f"{e['scale']['alpha']:g}/{e['scale']['beta']:g}/{e['scale']['gamma']:g}",
    'from': ', '.join(f"{k}<-{v}" for k, v in e['scale']['src'].items()
                      if k in ('nu', 'utau', 'dudy_ref', 'reward_mode')),
} for il, (case, e) in enumerate(case_dict.items())])
prov

### Plot-ready products

Everything the figures below draw, computed once per case. Section 8 archives
these very objects, so a saved `.mat` is the figure above it rather than a
recomputation that quietly used different bins.

`by_policy` breakdowns are added at write time for cases with more than one
`ipol` region — on the wing the four regions sit at different chord stations
and a global pool smears them.

In [ ]:
PRODUCTS = {
    case: {
        'reward_terms': dr.reward_terms(entry, TRANS_TIME, EVAL_TIME),
        'observation':  dr.observation_pdf(entry, nbin=N_MESH),
        'action':       dr.action_traces(entry, n_show=N_AGENT_SHOW),
        'control_law':  dr.control_law(entry, nbin=N_BIN, min_count=MIN_COUNT),
        'sensitivity':  dr.sensitivity_sweep(entry),
    }
    for case, entry in case_dict.items()
}

for case, p in PRODUCTS.items():
    filled = int(np.isfinite(p['control_law']['act_mean']).sum())
    print(f"{case:<22} regions {dr.policies(case_dict[case])}  "
          f"obs {p['observation']['nsample']:,d} samples  "
          f"law {filled}/{N_BIN ** 2} bins filled  "
          f"traces {p['action']['traces'].shape}")

## 1. Reward

Ensemble view: the line is the mean over the envs of a case, the band its
spread. With one env there is no band.

In [ ]:
fig, axs = plt.subplots(1, 1, figsize=(9, 6))
for il, (case, entry) in enumerate(case_dict.items()):
    style = copy.deepcopy(case_style[il])
    rt = PRODUCTS[case]['reward_terms']
    tp, m, s = rt['tp'], rt['R_tot_mean'], rt['R_tot_std']
    row = dr.case_summary(entry, TRANS_TIME, EVAL_TIME)
    if entry['nenv'] > 1:
        axs.fill_between(tp, m - s, m + s, color=style['c'], alpha=0.25, lw=0)
    style['label'] = f"{case_name[il]}: {row['R_tot']:+.3f}"
    style['markevery'] = max(1, len(tp) // 12)
    axs.plot(tp, m, **style)

if XMAX > TRANS_TIME:
    axs.axvspan(0, TRANS_TIME, color='gray', alpha=0.2)
    axs.text(0.02, 0.90, "Transient:\n" + rf"$t^+ \leq {TRANS_TIME:.0f}$",
             transform=axs.transAxes, fontsize=12)
    axs.text(0.40, 0.90, "Reward evaluation:\n" + rf"$t^+ > {TRANS_TIME:.0f}$",
             transform=axs.transAxes, fontsize=12)
axs.axhline(0.0, color=cc.grays, lw=1, ls=(0, (4, 3)))
axs.set(xlabel=XLABEL, ylabel=r"$R$", xlim=[0, XMAX])
axs.legend(prop={"size": 14}, loc='best')
fig.tight_layout()
if SAVE_IMG:
    plt.savefig(fig_path + f'{HEAD}_Reward.jpg', dpi=300)
    print(f'Saved {fig_path}{HEAD}_Reward.jpg')
plt.show()

### Reward components

Unweighted, so they can be compared across runs that used different gains.
In `dudy` mode the solver never fills the two penalty slots and both rows are
identically zero.

These are averaged over every controlled agent. On the wing the policy regions
sit at different chord stations and that average smears them —
`dr.terms_by_policy(rec, scale)` returns the same decomposition one entry per
`ipol` when the regions need to be read apart.

In [ ]:
TERMS = [('R_tau', r"$R_\tau = 1-\tau_w/\tau_{ref}$"),
         ('R_pw',  r"$R_{pw} = -|p'v|/\tau_{ref}$"),
         ('R_v3',  r"$R_{v^3} = -0.5|v^3|/\tau_{ref}$")]

fig, axs = plt.subplots(len(TERMS), 1, figsize=(9, 10), sharex=True)
for ir, (key, ylab) in enumerate(TERMS):
    for il, (case, entry) in enumerate(case_dict.items()):
        style = copy.deepcopy(case_style[il])
        rt = PRODUCTS[case]['reward_terms']
        tp, m, s = rt['tp'], rt[f'{key}_mean'], rt[f'{key}_std']
        if entry['nenv'] > 1:
            axs[ir].fill_between(tp, m - s, m + s, color=style['c'],
                                 alpha=0.25, lw=0)
        style['label'] = case_name[il]
        style['markevery'] = max(1, len(tp) // 12)
        axs[ir].plot(tp, m, **style)
    axs[ir].axhline(0.0, color=cc.grays, lw=1, ls=(0, (4, 3)))
    axs[ir].set(ylabel=ylab, xlim=[0, XMAX])
    axs[ir].yaxis.label.set_size(18)
axs[-1].set(xlabel=XLABEL)
axs[0].legend(prop={"size": 14}, loc='best')
fig.tight_layout()
if SAVE_IMG:
    plt.savefig(fig_path + f'{HEAD}_RewardTerms.jpg', dpi=300)
    print(f'Saved {fig_path}{HEAD}_RewardTerms.jpg')
plt.show()

## 2. Drag reduction

The headline number: `100 (1 - tau_w/tau_ref)`, or `100 (1 - dUdy/dUdy_ref)` in
`dudy` mode. Faint line instantaneous, solid line the running mean; the legend
carries the mean over the evaluation window.

In [ ]:
fig, axs = plt.subplots(1, 1, figsize=(9, 6))
for il, (case, entry) in enumerate(case_dict.items()):
    style = copy.deepcopy(case_style[il])
    rt = PRODUCTS[case]['reward_terms']
    tp, m = rt['tp'], rt['dr_t_mean']
    row = dr.case_summary(entry, TRANS_TIME, EVAL_TIME)
    axs.plot(tp, m, color=style['c'], lw=1.2, alpha=0.30)
    style['label'] = f"{case_name[il]}: {row['DR_eval_pct']:+.2f}" + r"$\%$"
    style['markevery'] = max(1, len(tp) // 12)
    axs.plot(tp, np.cumsum(m) / np.arange(1, len(m) + 1), **style)

if XMAX > TRANS_TIME:
    axs.axvspan(0, TRANS_TIME, color='gray', alpha=0.2)
axs.axhline(0.0, color=cc.grays, lw=1, ls=(0, (4, 3)))
axs.set(xlabel=XLABEL, ylabel=r"$DR\ [\%]$", xlim=[0, XMAX])
axs.legend(prop={"size": 14}, loc='best')
fig.tight_layout()
if SAVE_IMG:
    plt.savefig(fig_path + f'{HEAD}_DragReduction.jpg', dpi=300)
    print(f'Saved {fig_path}{HEAD}_DragReduction.jpg')
plt.show()

## 3. Action

Exact traces — no averaging over envs. A few agents per case, one row per case
because agent identity does not carry across runs. The grey band is the
across-agent envelope of the case.

In [ ]:
agent_colors = [cc.blue, cc.red, cc.deepgreen, cc.deeppurple, cc.brown]
env_ls = ['-', '--', ':', '-.']

fig, axs = plt.subplots(len(case_dict), 1, figsize=(9, 3.2 * len(case_dict)),
                        sharex=True)
axs = np.atleast_1d(axs)
for il, (case, entry) in enumerate(case_dict.items()):
    p = PRODUCTS[case]['action']
    tp, tr = p['tp'], p['traces']            # (nenv, nt, nshown)
    for ie in range(tr.shape[0]):
        m, s = p['band_mean'][ie], p['band_std'][ie]
        axs[il].fill_between(tp, m - s, m + s, color=cc.grays, alpha=0.35, lw=0,
                             label=r'$\pm$1 std over agents' if ie == 0 else None)
        for ja in range(tr.shape[2]):
            axs[il].plot(tp, tr[ie, :, ja],
                         color=agent_colors[ja % len(agent_colors)],
                         ls=env_ls[ie % len(env_ls)], lw=1.6, alpha=0.85,
                         label=f"agent {p['agent_index'][ja]}" if ie == 0 else None)
    axs[il].axhline(0.0, color=cc.grays, lw=1, ls=(0, (4, 3)))
    axs[il].set(ylabel=r"$a_i$", xlim=[0, XMAX], title=case_name[il])
    axs[il].title.set_size(18)
    axs[il].margins(y=0.30)          # room for the legend to land in
    axs[il].legend(prop={"size": 11}, loc='upper right', ncol=2,
                   framealpha=0.85)
axs[-1].set(xlabel=XLABEL)
fig.tight_layout()
if SAVE_IMG:
    plt.savefig(fig_path + f'{HEAD}_Action.jpg', dpi=300)
    print(f'Saved {fig_path}{HEAD}_Action.jpg')
plt.show()

### Action distribution

Every recorded sample of every agent and every env, pooled — exact, nothing
averaged. Mass piled at the bounds means `tanh` is railed and the policy is
effectively bang-bang.

In [ ]:
fig, axs = plt.subplots(1, 1, figsize=(9, 6))
for il, (case, entry) in enumerate(case_dict.items()):
    p = PRODUCTS[case]['action']
    axs.stairs(p['pdf'], p['pdf_edges'], color=case_style[il]['c'], lw=2.2,
               label=f"{case_name[il]}: {p['act_sat_pct']:.1f}" + r"$\%$ sat.")
axs.set(xlabel=r"$a_i$", ylabel="PDF")
axs.legend(prop={"size": 14}, loc='best')
fig.tight_layout()
if SAVE_IMG:
    plt.savefig(fig_path + f'{HEAD}_ActionPDF.jpg', dpi=300)
    print(f'Saved {fig_path}{HEAD}_ActionPDF.jpg')
plt.show()

## 4. Observation

Two components at the sensing plane: wall-tangential and wall-normal
fluctuations, divided by `u_tau`. Pooled exactly across agents and envs.

The joint density is a 2-D histogram rather than a KDE — at 10^5–10^7 samples a
Gaussian KDE is not tractable, and at that count the histogram is already smooth.

In [ ]:
fig, axs = plt.subplots(1, len(case_dict), figsize=(5.4 * len(case_dict), 5),
                        squeeze=False)
axs = axs[0]
for il, (case, entry) in enumerate(case_dict.items()):
    p = PRODUCTS[case]['observation']
    im = axs[il].pcolormesh(p['xedges'], p['yedges'], p['pdf'].T, cmap=SEQ,
                            shading='flat', rasterized=True)
    axs[il].set(title=case_name[il], xlabel=obs_lab[0],
                ylabel=obs_lab[1] if il == 0 else None)
    axs[il].title.set_size(18)
    cb = fig.colorbar(im, ax=axs[il], pad=0.02)
    cb.set_label('PDF', size=14)
    cb.ax.tick_params(labelsize=12)
fig.tight_layout()
if SAVE_IMG:
    plt.savefig(fig_path + f'{HEAD}_ObservationJoint.jpg', dpi=300)
    print(f'Saved {fig_path}{HEAD}_ObservationJoint.jpg')
plt.show()

### Marginals

The same samples, one panel per component, cases overlaid.

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(12, 5))
for il, (case, entry) in enumerate(case_dict.items()):
    p = PRODUCTS[case]['observation']
    for k in range(2):
        axs[k].stairs(p[f'pdf{k + 1}'], p[f'edges{k + 1}'], lw=2.2,
                      color=case_style[il]['c'],
                      label=case_name[il] if k == 0 else None)
for k in range(2):
    axs[k].set(xlabel=obs_lab[k], ylabel="PDF" if k == 0 else None)
axs[0].legend(prop={"size": 14}, loc='best')
fig.tight_layout()
if SAVE_IMG:
    plt.savefig(fig_path + f'{HEAD}_ObservationPDF.jpg', dpi=300)
    print(f'Saved {fig_path}{HEAD}_ObservationPDF.jpg')
plt.show()

## 5. The learned control law

Binning the recorded action on the observation plane recovers the policy that
actually flew, straight from the trajectory — the complete input-output map,
restricted to the states the flow visited. Opposition control would be the
plane `a = -v'`: a horizontal stripe pattern with no `u'` dependence.

One panel per case; the colour scale is shared so the panels can be compared.

Speckle means thin sampling, not structure: at `MIN_COUNT = 1` a bin holding
one sample is drawn as confidently as one holding a thousand, and a single
railed action paints a saturated cell. Raise `MIN_COUNT` to blank them, or read
`counts` in the saved `.mat`, which carries the sample count of every bin.

In [ ]:
laws = {case: PRODUCTS[case]['control_law'] for case in case_dict}
vmax = max(np.nanmax(np.abs(p['act_mean'])) for p in laws.values())

fig, axs = plt.subplots(1, len(case_dict), figsize=(5.4 * len(case_dict), 5),
                        squeeze=False)
axs = axs[0]
for il, (case, entry) in enumerate(case_dict.items()):
    p = laws[case]
    im = axs[il].pcolormesh(p['xedges'], p['yedges'], p['act_mean'].T, cmap=DIV,
                            vmin=-vmax, vmax=vmax, shading='flat',
                            rasterized=True)
    axs[il].set(title=case_name[il], xlabel=obs_lab[0],
                ylabel=obs_lab[1] if il == 0 else None)
    axs[il].title.set_size(18)
cb = fig.colorbar(im, ax=axs, pad=0.02)
cb.set_label(r"$\langle a_i \rangle$", size=16)
cb.ax.tick_params(labelsize=12)
if SAVE_IMG:
    plt.savefig(fig_path + f'{HEAD}_ControlLaw.jpg', dpi=300,
                bbox_inches='tight')
    print(f'Saved {fig_path}{HEAD}_ControlLaw.jpg')
plt.show()

### Sensitivity to each input

One component swept with the other held near its median, cases overlaid.
A pure opposition controller is flat in `u_t^+` and a straight negative slope
in `v_n^+`.

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(12, 5))
for il, (case, entry) in enumerate(case_dict.items()):
    style = copy.deepcopy(case_style[il])
    p = PRODUCTS[case]['sensitivity']
    for k in range(2):
        xc, ac = p[f'x{k + 1}'], p[f'a{k + 1}']
        if xc.size == 0:
            print(f"[WARN] {case_name[il]}: only {p[f'n{k + 1}']} samples near "
                  f"the median of {obs_lab[1 - k]}; sweep skipped")
            continue
        axs[k].plot(xc, ac, color=style['c'], marker=style['marker'],
                    markersize=6, lw=2,
                    label=case_name[il] if k == 0 else None)
for k in range(2):
    axs[k].axhline(0.0, color=cc.grays, lw=1, ls=(0, (4, 3)))
    axs[k].set(xlabel=obs_lab[k],
               ylabel=r"$\langle a_i \rangle$" if k == 0 else None)
axs[0].legend(prop={"size": 14}, loc='best')
fig.suptitle("swept one component at a time, the other held near its median",
             fontsize=15)
fig.tight_layout()
if SAVE_IMG:
    plt.savefig(fig_path + f'{HEAD}_Sensitivity.jpg', dpi=300)
    print(f'Saved {fig_path}{HEAD}_Sensitivity.jpg')
plt.show()

## 6. Agent layout

The identity block travels inside every record file, so the ranks rejoin
without a `NODE_INFO.csv` and without knowing the mesh partition. Colour is the
policy region: `ipol = 0` is wall that no policy claims, and is what
`agents='active'` drops.

In [ ]:
fig, axs = plt.subplots(1, len(case_dict), figsize=(5.6 * len(case_dict), 5.2),
                        squeeze=False)
axs = axs[0]
for il, (case, entry) in enumerate(case_dict.items()):
    ag = entry['envs'][0]['rec'].agents
    pol = ag['ipol'].to_numpy()
    for ip in np.unique(pol):
        m = pol == ip
        axs[il].scatter(ag['x'][m], ag['z'][m], s=16,
                        color=cc.grays if ip == 0
                        else agent_colors[(int(ip) - 1) % len(agent_colors)],
                        label=f"ipol {ip}" + (" (idle)" if ip == 0 else ""),
                        edgecolor='white', linewidth=0.3)
    axs[il].set(xlabel=r"$x$", ylabel=r"$z$" if il == 0 else None,
                title=case_name[il])
    axs[il].title.set_size(18)
    if len(np.unique(pol)) > 1:
        # The wall is covered edge to edge, so the legend has to sit outside it.
        axs[il].legend(prop={"size": 11}, ncol=3, loc='upper center',
                       bbox_to_anchor=(0.5, -0.42), frameon=False)
fig.tight_layout()
if SAVE_IMG:
    plt.savefig(fig_path + f'{HEAD}_AgentLayout.jpg', dpi=300)
    print(f'Saved {fig_path}{HEAD}_AgentLayout.jpg')
plt.show()

## 7. Summary

`window` says which records the means were taken over: a run too short to reach
the evaluation window falls back to the whole record and reports that, rather
than quietly averaging the start-up transient into a "converged" number.

In [ ]:
summary = dr.summary_table(case_dict, TRANS_TIME, EVAL_TIME, labels=case_name)
if SAVE_IMG:
    summary.to_csv(fig_path + f'{HEAD}_summary.csv', index=False)
    print(f'Saved {fig_path}{HEAD}_summary.csv')
summary

## 8. Export

### 8.1 The full record

One `.mat` per env into the results tree, beside what `utils/collect-results`
and `tsrs_spectra.py --results` write for the same run:

```
data/results/<solver_case>/<run_name>/
    config/  current_conf.yml, env<NNN>_drl_policy.in
    drl/     <run_name>_env<NNN>_drlrec.mat
    meta.yml
```

Each file is flat and self-contained: `act`, `obs` and `rwd` per agent, the
agent-averaged series, every reward term as its own variable, the agent table
(with `used` flagging the agents the series were built from), and the scalars it
was all derived with — so it can be re-scaled later without the original run.

`save_full=False` drops the three big per-agent arrays and keeps everything else.

In [ ]:
SAVE_FULL = True
written = dr.save_all(case_dict, save_full=SAVE_FULL,
                      trans_time=TRANS_TIME, eval_time=EVAL_TIME)
written

### 8.2 Plot-ready products

The four figures above, as the arrays that made them, one flat `.mat` each:

```
data/results/<solver_case>/<run_name>/drl/processed_data/
    <run_name>_reward_terms.mat   every term per env, plus the ensemble mean/std
    <run_name>_observation.mat    X, Y, PDF of the observation plane + marginals
    <run_name>_action.mat         traces of a few agents, envelope, action PDF
    <run_name>_control_law.mat    X, Y, <a> over the plane + the 1-D sweeps
```

All four for one case come to well under a megabyte, against gigabytes for the
per-agent arrays in 8.1 — these are the files to hand to someone who wants the
figure rather than the run.

`precomputed=PRODUCTS` archives the very objects plotted above, so a file can
never be a recomputation that used different bins. Each carries the same
`scale` and `meta` blocks as the full record, so it stands on its own.

`X`, `Y` and the value array are the same shape and go straight into
`pcolor`/`surf`/`contourf`. `counts` sits beside every histogram: a density
says nothing about how many samples produced it, and the sparse corners of the
observation plane are where that matters.

Cases with more than one `ipol` region also get a `polN` struct per region,
holding the same fields — on the wing the four regions sit at different chord
stations and the pooled version smears them.

In [ ]:
light = dr.save_light(case_dict, precomputed=PRODUCTS,
                      by_policy='auto', trans_time=TRANS_TIME,
                      eval_time=EVAL_TIME)
light

In [ ]:
# Reading one back, in Python:
# import scipy.io as sio
# d = sio.loadmat(path, squeeze_me=True, struct_as_record=False)
# d['X'], d['Y'], d['act_mean']          # the control-law surface
# d['pol2'].act_mean                     # one policy region, where present
# d['meta'].note                         # what the variables in this file mean
#
# squeeze_me=True drops singleton axes, so with one env `R_tot` reads back as
# (nt,) rather than (1, nt). Pass squeeze_me=False to keep the env axis.
#
# In MATLAB:
#   d = load(path);  pcolor(d.X, d.Y, d.act_mean);  shading flat

In [ ]:
# Reading one back:
# import scipy.io as sio
# d = sio.loadmat(path, squeeze_me=True, struct_as_record=False)
# d['act'], d['obs'], d['R_tot'], d['scale'].nu, d['meta'].eval_window